# 05 Google Sheets Sync + Ops

Run end-to-end process, push rows to Google Sheets, export CSV/Excel, delete source on success.

In [ ]:
from pathlib import Path
import sys
sys.path.append(str(Path.cwd() / 'src'))
from rail_video_intelligence.pipeline import (
    RailVideoPipeline,
    SheetsConfig,
    load_camera_profile,
    load_pipeline_settings,
    parse_pipeline_settings,
)

VIDEO_PATH = Path('data/raw/camera_01_2026_04_09_233235.MOV')
PIPELINE_CONFIG = Path('configs/pipeline.yaml')
PROFILE_PATH = Path('configs/camera_profiles/camera_01.yaml')

In [ ]:
raw = load_pipeline_settings(PIPELINE_CONFIG)
settings = parse_pipeline_settings(raw.get('pipeline', raw))
sheets_raw = raw.get('sheets', {})
sheets = SheetsConfig(
    enabled=bool(sheets_raw.get('enabled', False)),
    spreadsheet_id=sheets_raw.get('spreadsheet_id'),
    worksheet_name=sheets_raw.get('worksheet_name', 'train_events'),
    credentials_path=Path(sheets_raw['credentials_path']) if sheets_raw.get('credentials_path') else None,
    retries=int(sheets_raw.get('retries', 4)),
    backoff_seconds=float(sheets_raw.get('backoff_seconds', 1.5)),
)

pipeline = RailVideoPipeline(
    settings=settings,
    camera_profile=load_camera_profile(PROFILE_PATH),
    output_root=Path(raw.get('output_root', 'outputs/v1')),
    sheets_config=sheets,
)

result = pipeline.process_video(
    video_path=VIDEO_PATH,
    run_name='nb_ops',
    delete_on_success=True,
    export_excel=True,
)
result